# 110 — Offline P&P accept/reject audit

Scores the exact nine recorded v4 branches at each held-out root. Candidate 0 is the stored P&P source; candidates 1–8 are fresh-noise P&P proposals. This measures **one intervention under P&P continuation**, not an online deployed selector. Threshold selection on part of the validation roots is exploratory; use new episode indices for a final claim.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXTRAS = 'sim'
SETUP_ENV = False
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import torch
from pnp.smolvla_q_selection import (
    FRESH8_EXPERIMENT, FRESH8_KINDS, fresh8_snapshot_key,
    load_success_q10, score_root_trees, paired_counts)
from pnp.smolvla_success_critic import TimedRoots
from pnp.smolvla_tree_bellman_finetune import load_or_create_tree_snapshot, prepare_tree_bellman_cache
from pnp.store import SupabaseStore

TREE_LIMIT = 800
CHECKPOINT = Path('REPLACE_WITH_NOTEBOOK_109_FINAL_CHECKPOINT.pt')
ROOT_TOKEN_DIR = None  # set only for notebook 111's RLT-Q checkpoint
CACHE_ROOT = Path('/content/smolvla_q10_fresh8_cache')
assert CHECKPOINT.is_file(), 'Set CHECKPOINT to a saved success-Q10 checkpoint.'
store = SupabaseStore()
snapshot = load_or_create_tree_snapshot(
    store=store, snapshot_key=fresh8_snapshot_key(TREE_LIMIT), tree_limit=TREE_LIMIT,
    experiment=FRESH8_EXPERIMENT, candidate_kinds=FRESH8_KINDS)
cache = prepare_tree_bellman_cache(
    snapshot=snapshot, cache_root=CACHE_ROOT, gamma=1.0,
    success_reward=True, store=store)
model, payload = load_success_q10(
    CHECKPOINT, expected_snapshot=snapshot['snapshot_digest'], device='cuda')
groups = {g['candidate_group_id']: g for g in snapshot['groups']}
roots = TimedRoots(cache, cache['validation_group_ids'], groups, ROOT_TOKEN_DIR)
scores, outcomes = score_root_trees(model, roots, device='cuda')
ids = [e['candidate_group_id'] for e in roots.entries]
assert outcomes.shape == scores.shape == (len(roots), 9)
print({'representation': payload.get('representation'), 'roots': len(roots),
       'baseline': paired_counts(outcomes, np.zeros(len(roots), dtype=int))})


In [ ]:
# Development/test division is fixed by root identity. Neither part is a fresh final test:
# training checkpoints have already been inspected on this validation cohort.
order = sorted(range(len(ids)), key=lambda i: hashlib.sha256(ids[i].encode()).hexdigest())
dev = np.asarray(order[:len(order)//2]); test = np.asarray(order[len(order)//2:])
best = scores[:, 1:].argmax(1) + 1
delta = scores[np.arange(len(scores)), best] - scores[:, 0]
thresholds = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30]
def selected(tau):
    return np.where(delta > tau, best, 0)
development = [(tau, paired_counts(outcomes[dev], selected(tau)[dev])) for tau in thresholds]
for tau, metrics in development:
    print('development', tau, metrics)
tau = max(thresholds, key=lambda t: (
    paired_counts(outcomes[dev], selected(t)[dev])['selected_successes'], t))
print('chosen development threshold:', tau)
print('exploratory held-out half:', paired_counts(outcomes[test], selected(tau)[test]))
print('ungated argmax:', paired_counts(outcomes, scores.argmax(1)))
